# 🏗️ ETL Pipeline — Módulo Prático
## Parte 1: Ingestão, Camadas Bronze/Silver/Gold e Qualidade de Dados
### Usando Python + Pandas

---

> **Objetivo:** Construir um pipeline ETL completo, desde a ingestão de dados até a entrega de dados prontos para análise, aplicando as boas práticas de qualidade de dados e arquitetura em camadas (Medallion Architecture).

---

## 📌 Estrutura da Demo

| Etapa | Descrição |
|---|---|
| **1. Setup** | Instalação de dependências e configuração do ambiente |
| **2. Ingestão (Bronze)** | Consumo de API pública + armazenamento raw |
| **3. Limpeza (Silver)** | Tratamento, normalização e deduplicação |
| **4. Enriquecimento (Gold)** | Agregações, métricas e dados prontos para consumo |
| **5. Qualidade de Dados** | Validações, regras e relatório de qualidade |

---
## ⚙️ Etapa 1 — Setup do Ambiente

In [ ]:
# ============================================================
# ETAPA 1 — INSTALAÇÃO DE DEPENDÊNCIAS
# ============================================================
# Instalamos as bibliotecas necessárias para o pipeline.
# 'great_expectations' é uma das principais libs de qualidade de dados.

!pip install great_expectations pandas requests colorama tabulate --quiet

print("Dependências instaladas com sucesso!")

In [ ]:
# ============================================================
# IMPORTS E CONFIGURAÇÕES GLOBAIS
# ============================================================
import pandas as pd
import numpy as np
import requests
import json
import os
import hashlib
from datetime import datetime, timezone
from colorama import Fore, Style, init
from tabulate import tabulate
import warnings
warnings.filterwarnings('ignore')

init(autoreset=True)

# ─── Configuração de Caminhos (simula um Data Lake local) ───
BASE_PATH = "/content/data_lake"
BRONZE_PATH = f"{BASE_PATH}/bronze"
SILVER_PATH = f"{BASE_PATH}/silver"
GOLD_PATH   = f"{BASE_PATH}/gold"
QUALITY_PATH = f"{BASE_PATH}/quality_reports"

for path in [BRONZE_PATH, SILVER_PATH, GOLD_PATH, QUALITY_PATH]:
    os.makedirs(path, exist_ok=True)

INGESTION_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

print(f"{Fore.GREEN} Ambiente configurado!")
print(f"{Fore.CYAN} Estrutura do Data Lake:")
print(f"   {BASE_PATH}/")
print(f"   ├── bronze/   → Dados brutos (raw)")
print(f"   ├── silver/   → Dados limpos e normalizados")
print(f"   ├── gold/     → Dados agregados para consumo")
print(f"   └── quality_reports/ → Relatórios de qualidade")
print(f"\n{Fore.YELLOW} Timestamp de ingestão: {INGESTION_TIMESTAMP}")

---
## 🥉 Etapa 2 — Camada BRONZE: Ingestão de Dados

A **camada Bronze** armazena os dados **exatamente como vieram da fonte** — sem nenhuma transformação.
Isso garante rastreabilidade e a possibilidade de reprocessamento.

**Fonte utilizada:** [JSONPlaceholder](https://jsonplaceholder.typicode.com/) — API pública gratuita que simula dados de e-commerce/usuários.

> 💡 **Boa prática:** Sempre adicionar metadados de ingestão (timestamp, fonte, versão) aos dados brutos.

In [ ]:
# ============================================================
# ETAPA 2.1 — FUNÇÃO DE INGESTÃO GENÉRICA
# ============================================================
# Criamos uma função reutilizável para ingestão via HTTP.
# Ela trata erros, adiciona metadados e salva os dados brutos.

def ingest_from_api(endpoint: str, entity_name: str) -> pd.DataFrame:
    """
    Ingere dados de uma API REST e salva na camada Bronze.

    Args:
        endpoint: URL da API
        entity_name: Nome da entidade (ex: 'users', 'orders')

    Returns:
        DataFrame com os dados brutos + metadados
    """
    print(f"{Fore.CYAN} Iniciando ingestão de [{entity_name}]...")
    print(f"   URL: {endpoint}")

    try:
        response = requests.get(endpoint, timeout=10)
        response.raise_for_status()  # Lança exceção se status != 200

        raw_data = response.json()
        df = pd.DataFrame(raw_data)

        # ── Adicionando metadados de ingestão ──
        df['_ingestion_timestamp'] = INGESTION_TIMESTAMP
        df['_source_url']          = endpoint
        df['_source_system']       = 'jsonplaceholder_api'
        df['_record_hash'] = df.drop(
            columns=['_ingestion_timestamp', '_source_url', '_source_system'],
            errors='ignore'
        ).apply(lambda row: hashlib.md5(str(row.values).encode()).hexdigest(), axis=1)

        # ── Salvando na camada Bronze ──
        output_file = f"{BRONZE_PATH}/{entity_name}_raw_{INGESTION_TIMESTAMP}.parquet"
        df.to_parquet(output_file, index=False)

        print(f"{Fore.GREEN}    {len(df)} registros ingeridos")
        print(f"    Salvo em: {output_file}")
        print(f"    Colunas: {list(df.columns)}")

        return df

    except requests.exceptions.Timeout:
        print(f"{Fore.RED}    ERRO: Timeout na requisição")
        raise
    except requests.exceptions.HTTPError as e:
        print(f"{Fore.RED}    ERRO HTTP: {e}")
        raise
    except Exception as e:
        print(f"{Fore.RED}    ERRO inesperado: {e}")
        raise

print(f"{Fore.GREEN} Função de ingestão definida!")

In [ ]:
# ============================================================
# ETAPA 2.2 — INGESTÃO MULTI-ENTIDADE
# ============================================================
# Ingerimos 3 entidades que se relacionam entre si:
# - users   → cadastro de clientes
# - posts   → simula pedidos/transações
# - todos   → simula status de atividades

print(f"{Fore.YELLOW}{'='*60}")
print(f"{Fore.YELLOW}  INICIANDO PIPELINE DE INGESTÃO")
print(f"{Fore.YELLOW}{'='*60}\n")

BASE_URL = "https://jsonplaceholder.typicode.com"

# Dicionário com todas as entidades a ingerir
entities = {
    'users':   f"{BASE_URL}/users",
    'posts':   f"{BASE_URL}/posts",
    'todos':   f"{BASE_URL}/todos",
    'comments': f"{BASE_URL}/comments",
}

bronze_dfs = {}
for entity, url in entities.items():
    bronze_dfs[entity] = ingest_from_api(url, entity)
    print()

print(f"{Fore.GREEN}{'='*60}")
print(f"{Fore.GREEN}  INGESTÃO CONCLUÍDA")
print(f"{Fore.GREEN}   Total de entidades: {len(bronze_dfs)}")
print(f"{Fore.GREEN}   Total de registros: {sum(len(df) for df in bronze_dfs.values())}")
print(f"{Fore.GREEN}{'='*60}")

In [ ]:
# ============================================================
# ETAPA 2.3 — INSPEÇÃO DOS DADOS BRUTOS
# ============================================================
# Sempre inspecionar os dados antes de processá-los!

print(f"{Fore.YELLOW} PRÉVIA DOS DADOS BRUTOS (Bronze)\n")

for entity, df in bronze_dfs.items():
    print(f"{Fore.CYAN}{'─'*50}")
    print(f"{Fore.CYAN}  Entidade: {entity.upper()}")
    print(f"   Shape: {df.shape} | Nulos totais: {df.isnull().sum().sum()}")
    print(f"   Tipos: { {col: str(dtype) for col, dtype in df.dtypes.items()} }")
    print()
    display(df.head(3))
    print()

---
## 🥈 Etapa 3 — Camada SILVER: Limpeza e Normalização

A **camada Silver** aplica transformações para tornar os dados **confiáveis e consistentes**:
- Tipagem correta
- Remoção de duplicatas
- Normalização de texto
- Tratamento de nulos
- Parsing de campos aninhados (JSON dentro de JSON)

> 💡 **Boa prática:** Nunca modificar os dados Bronze. Sempre partir deles para criar a Silver.

In [ ]:
# ============================================================
# ETAPA 3.1 — TRANSFORMAÇÃO: USERS
# ============================================================

print(f"{Fore.CYAN} Processando entidade: USERS\n")

df_users_bronze = bronze_dfs['users'].copy()

# ── 1. Flatten de colunas aninhadas (address, company, geo) ──
print("  → Expandindo colunas aninhadas (address, company, geo)...")

address_df = pd.json_normalize(df_users_bronze['address'])
address_df.columns = ['addr_' + col.lower().replace('.', '_') for col in address_df.columns]

company_df = pd.json_normalize(df_users_bronze['company'])
company_df.columns = ['company_' + col.lower().replace(' ', '_') for col in company_df.columns]

df_users_silver = pd.concat([
    df_users_bronze[['id', 'name', 'username', 'email', 'phone', 'website',
                     '_ingestion_timestamp', '_source_system', '_record_hash']],
    address_df,
    company_df
], axis=1)

# ── 2. Normalização de texto ──
print("  → Normalizando campos de texto (strip, lower onde aplicável)...")
df_users_silver['email']    = df_users_silver['email'].str.strip().str.lower()
df_users_silver['username'] = df_users_silver['username'].str.strip().str.lower()
df_users_silver['name']     = df_users_silver['name'].str.strip().str.title()
df_users_silver['phone']    = df_users_silver['phone'].str.replace(r'[^\d\+\-\(\) x]', '', regex=True)

# ── 3. Renomear colunas para padrão snake_case ──
df_users_silver.rename(columns={
    'id': 'user_id',
    'addr_geo_lat': 'geo_lat',
    'addr_geo_lng': 'geo_lng',
}, inplace=True)

# ── 4. Tipagem correta ──
print("  → Aplicando tipos corretos...")
df_users_silver['user_id'] = df_users_silver['user_id'].astype(int)
df_users_silver['geo_lat'] = pd.to_numeric(df_users_silver['geo_lat'], errors='coerce')
df_users_silver['geo_lng'] = pd.to_numeric(df_users_silver['geo_lng'], errors='coerce')

# ── 5. Deduplicação ──
duplicates_before = len(df_users_silver)
df_users_silver = df_users_silver.drop_duplicates(subset=['user_id'])
print(f"  → Duplicatas removidas: {duplicates_before - len(df_users_silver)}")

# ── 6. Adicionar metadado de processamento ──
df_users_silver['_silver_processed_at'] = datetime.now(timezone.utc).isoformat()
df_users_silver['_pipeline_version']    = 'v1.0'

# ── 7. Salvar na camada Silver ──
df_users_silver.to_parquet(f"{SILVER_PATH}/users_silver.parquet", index=False)

print(f"\n{Fore.GREEN}   Users Silver: {df_users_silver.shape[0]} registros, {df_users_silver.shape[1]} colunas")
display(df_users_silver.head(3))

In [ ]:
# ============================================================
# ETAPA 3.2 — TRANSFORMAÇÃO: POSTS (simula transações/pedidos)
# ============================================================

print(f"{Fore.CYAN} Processando entidade: POSTS (transações)\n")

df_posts_bronze = bronze_dfs['posts'].copy()

# ── 1. Renomear para contexto de negócio ──
df_posts_silver = df_posts_bronze[['userId', 'id', 'title', 'body',
                                    '_ingestion_timestamp', '_source_system', '_record_hash']].copy()
df_posts_silver.rename(columns={
    'userId': 'user_id',
    'id': 'post_id',
}, inplace=True)

# ── 2. Normalização de texto ──
df_posts_silver['title'] = df_posts_silver['title'].str.strip().str.title()
df_posts_silver['body']  = df_posts_silver['body'].str.strip().str.replace('\n', ' ', regex=False)

# ── 3. Criar campos derivados ──
print("  → Criando campos derivados...")
df_posts_silver['title_word_count'] = df_posts_silver['title'].str.split().str.len()
df_posts_silver['body_word_count']  = df_posts_silver['body'].str.split().str.len()
df_posts_silver['body_length']      = df_posts_silver['body'].str.len()

# ── 4. Tipagem ──
df_posts_silver['user_id'] = df_posts_silver['user_id'].astype(int)
df_posts_silver['post_id'] = df_posts_silver['post_id'].astype(int)

# ── 5. Deduplicação ──
df_posts_silver = df_posts_silver.drop_duplicates(subset=['post_id'])

df_posts_silver['_silver_processed_at'] = datetime.now(timezone.utc).isoformat()
df_posts_silver.to_parquet(f"{SILVER_PATH}/posts_silver.parquet", index=False)

print(f"{Fore.GREEN}   Posts Silver: {df_posts_silver.shape}")
display(df_posts_silver.head(3))

In [ ]:
# ============================================================
# ETAPA 3.3 — TRANSFORMAÇÃO: TODOS (status de atividades)
# ============================================================

print(f"{Fore.CYAN} Processando entidade: TODOS (atividades)\n")

df_todos_bronze = bronze_dfs['todos'].copy()

df_todos_silver = df_todos_bronze[['userId', 'id', 'title', 'completed',
                                    '_ingestion_timestamp', '_source_system', '_record_hash']].copy()
df_todos_silver.rename(columns={
    'userId': 'user_id',
    'id': 'todo_id',
}, inplace=True)

# ── Normalização ──
df_todos_silver['title'] = df_todos_silver['title'].str.strip().str.title()

# ── Criar campo de status legível ──
df_todos_silver['status'] = df_todos_silver['completed'].map({True: 'COMPLETED', False: 'PENDING'})

# ── Tipagem ──
df_todos_silver['user_id'] = df_todos_silver['user_id'].astype(int)
df_todos_silver['todo_id'] = df_todos_silver['todo_id'].astype(int)
df_todos_silver['completed'] = df_todos_silver['completed'].astype(bool)

df_todos_silver['_silver_processed_at'] = datetime.now(timezone.utc).isoformat()
df_todos_silver.to_parquet(f"{SILVER_PATH}/todos_silver.parquet", index=False)

print(f"{Fore.GREEN}   Todos Silver: {df_todos_silver.shape}")
display(df_todos_silver.head(3))

---
## 🥇 Etapa 4 — Camada GOLD: Agregações e Métricas de Negócio

A **camada Gold** entrega dados **prontos para consumo** por ferramentas de BI, dashboards ou modelos de ML.
Aqui criamos:
- Visões consolidadas (joins entre entidades)
- Métricas agregadas por usuário
- Indicadores de negócio

> 💡 **Boa prática:** A Gold deve ser otimizada para leitura — menos colunas, mais semântica de negócio.

In [ ]:
# ============================================================
# ETAPA 4.1 — GOLD: PERFIL COMPLETO DO USUÁRIO
# ============================================================
# Join entre Users + Posts + Todos para criar um perfil 360°

print(f"{Fore.YELLOW} Construindo camada GOLD...\n")

# ── Métricas de posts por usuário ──
posts_agg = df_posts_silver.groupby('user_id').agg(
    total_posts          = ('post_id', 'count'),
    avg_title_words      = ('title_word_count', 'mean'),
    avg_body_length      = ('body_length', 'mean'),
).reset_index().round(2)

# ── Métricas de tarefas por usuário ──
todos_agg = df_todos_silver.groupby('user_id').agg(
    total_tasks       = ('todo_id', 'count'),
    completed_tasks   = ('completed', 'sum'),
    pending_tasks     = ('completed', lambda x: (~x).sum()),
).reset_index()

todos_agg['completion_rate_pct'] = (
    todos_agg['completed_tasks'] / todos_agg['total_tasks'] * 100
).round(1)

# ── Join completo ──
df_gold_user_profile = (
    df_users_silver[['user_id', 'name', 'email', 'username',
                     'addr_city', 'addr_zipcode', 'company_name',
                     'geo_lat', 'geo_lng']]
    .merge(posts_agg, on='user_id', how='left')
    .merge(todos_agg, on='user_id', how='left')
)

# ── Classificação de engajamento ──
def classify_engagement(row):
    if row['completion_rate_pct'] >= 70 and row['total_posts'] >= 10:
        return 'HIGH'
    elif row['completion_rate_pct'] >= 40 or row['total_posts'] >= 5:
        return 'MEDIUM'
    else:
        return 'LOW'

df_gold_user_profile['engagement_level'] = df_gold_user_profile.apply(classify_engagement, axis=1)

# ── Metadados Gold ──
df_gold_user_profile['_gold_processed_at'] = datetime.now(timezone.utc).isoformat()

# ── Salvar ──
df_gold_user_profile.to_parquet(f"{GOLD_PATH}/user_profile_gold.parquet", index=False)

print(f"{Fore.GREEN} Gold - User Profile: {df_gold_user_profile.shape}")
display(df_gold_user_profile)

In [ ]:
# ============================================================
# ETAPA 4.2 — GOLD: ANÁLISE POR CIDADE
# ============================================================

df_gold_city = df_gold_user_profile.groupby('addr_city').agg(
    total_users        = ('user_id', 'count'),
    total_posts        = ('total_posts', 'sum'),
    avg_completion_pct = ('completion_rate_pct', 'mean'),
    high_engagement    = ('engagement_level', lambda x: (x == 'HIGH').sum()),
).reset_index().round(2)

df_gold_city.columns.name = None
df_gold_city = df_gold_city.sort_values('avg_completion_pct', ascending=False)
df_gold_city['_gold_processed_at'] = datetime.now(timezone.utc).isoformat()

df_gold_city.to_parquet(f"{GOLD_PATH}/city_analysis_gold.parquet", index=False)

print(f"{Fore.GREEN} Gold - Análise por Cidade:")
display(df_gold_city)

# Sumário geral
print(f"\n{Fore.YELLOW} KPIs GLOBAIS:")
print(f"    Usuários únicos   : {df_gold_user_profile['user_id'].nunique()}")
print(f"    Total de posts    : {int(df_gold_user_profile['total_posts'].sum())}")
print(f"    Taxa conclusão avg: {df_gold_user_profile['completion_rate_pct'].mean():.1f}%")
print(f"    Alta eng. (HIGH)  : {(df_gold_user_profile['engagement_level'] == 'HIGH').sum()} usuários")

---
## ✅ Etapa 5 — Qualidade de Dados

Aplicamos **validações automatizadas** em cada camada para garantir a confiabilidade dos dados.

Usaremos uma abordagem com **regras customizadas** (e optionalmente Great Expectations):
- **Completude**: Campos obrigatórios não podem ser nulos
- **Unicidade**: Chaves primárias devem ser únicas
- **Validade de formato**: Emails, tipos numéricos
- **Referencial**: IDs devem existir em outras tabelas
- **Range**: Valores dentro de limites esperados

> 💡 **Boa prática:** Qualidade de dados deve ser verificada **em cada camada**, não apenas no final.

In [ ]:
# ============================================================
# ETAPA 5.1 — FRAMEWORK DE QUALIDADE DE DADOS
# ============================================================

class DataQualityCheck:
    """Framework simples para validação de qualidade de dados."""

    def __init__(self, df: pd.DataFrame, entity_name: str, layer: str):
        self.df = df
        self.entity_name = entity_name
        self.layer = layer
        self.results = []
        self.passed = 0
        self.failed = 0
        self.warnings = 0

    def _add_result(self, check_name, dimension, passed, details, severity='ERROR'):
        status = ' PASS' if passed else ('  WARN' if severity == 'WARNING' else ' FAIL')
        if passed:
            self.passed += 1
        elif severity == 'WARNING':
            self.warnings += 1
        else:
            self.failed += 1

        self.results.append({
            'check': check_name,
            'dimension': dimension,
            'status': status,
            'details': details,
            'severity': severity
        })

    def check_not_null(self, columns, severity='ERROR'):
        """Verifica ausência de nulos em colunas obrigatórias."""
        for col in columns:
            if col not in self.df.columns:
                self._add_result(f'not_null:{col}', 'COMPLETENESS',
                                 False, f'Coluna {col} não existe', severity)
                continue
            null_count = self.df[col].isnull().sum()
            null_pct   = null_count / len(self.df) * 100
            passed     = null_count == 0
            self._add_result(
                f'not_null:{col}', 'COMPLETENESS', passed,
                f'{null_count} nulos ({null_pct:.1f}%)', severity
            )
        return self

    def check_unique(self, columns, severity='ERROR'):
        """Verifica unicidade das chaves primárias."""
        for col in columns:
            if col not in self.df.columns:
                continue
            dup_count = self.df[col].duplicated().sum()
            passed    = dup_count == 0
            self._add_result(
                f'unique:{col}', 'UNIQUENESS', passed,
                f'{dup_count} duplicatas encontradas', severity
            )
        return self

    def check_regex(self, column, pattern, description, severity='WARNING'):
        """Valida formato de valores usando regex."""
        if column not in self.df.columns:
            return self
        invalid = self.df[column].dropna().str.match(pattern) == False
        invalid_count = invalid.sum()
        passed = invalid_count == 0
        self._add_result(
            f'regex:{column}', 'VALIDITY', passed,
            f'{invalid_count} registros com formato inválido ({description})', severity
        )
        return self

    def check_range(self, column, min_val=None, max_val=None, severity='WARNING'):
        """Verifica se valores estão dentro de um intervalo esperado."""
        if column not in self.df.columns:
            return self
        series = self.df[column].dropna()
        out_of_range = 0
        if min_val is not None:
            out_of_range += (series < min_val).sum()
        if max_val is not None:
            out_of_range += (series > max_val).sum()
        passed = out_of_range == 0
        self._add_result(
            f'range:{column}', 'VALIDITY', passed,
            f'{out_of_range} fora do intervalo [{min_val}, {max_val}]', severity
        )
        return self

    def check_referential_integrity(self, column, ref_df, ref_column, severity='ERROR'):
        """Verifica integridade referencial entre tabelas."""
        if column not in self.df.columns:
            return self
        orphans = ~self.df[column].isin(ref_df[ref_column])
        orphan_count = orphans.sum()
        passed = orphan_count == 0
        self._add_result(
            f'referential:{column}→{ref_column}', 'CONSISTENCY', passed,
            f'{orphan_count} registros órfãos (FK ausente)', severity
        )
        return self

    def check_row_count(self, min_rows, severity='ERROR'):
        """Verifica contagem mínima de linhas."""
        passed = len(self.df) >= min_rows
        self._add_result(
            'row_count', 'COMPLETENESS', passed,
            f'{len(self.df)} linhas (mínimo: {min_rows})', severity
        )
        return self

    def generate_report(self):
        """Gera relatório final e salva em JSON."""
        total_checks = self.passed + self.failed + self.warnings
        score = round(self.passed / total_checks * 100, 1) if total_checks > 0 else 0

        print(f"\n{'═'*60}")
        print(f"  RELATÓRIO DE QUALIDADE — {self.entity_name.upper()} [{self.layer}]")
        print(f"{'═'*60}")
        print(tabulate(
            [(r['status'], r['dimension'], r['check'], r['details']) for r in self.results],
            headers=['Status', 'Dimensão', 'Verificação', 'Detalhes'],
            tablefmt='rounded_outline'
        ))

        color = Fore.GREEN if score >= 80 else (Fore.YELLOW if score >= 60 else Fore.RED)
        print(f"\n{color} Score de Qualidade: {score}%")
        print(f"    Passed : {self.passed} |  Failed: {self.failed} |   Warnings: {self.warnings}")

        report = {
            'entity': self.entity_name, 'layer': self.layer,
            'score': score, 'timestamp': datetime.now(timezone.utc).isoformat(),
            'summary': {'passed': self.passed, 'failed': self.failed, 'warnings': self.warnings},
            'checks': self.results
        }
        report_path = f"{QUALITY_PATH}/{self.entity_name}_{self.layer}_quality.json"
        with open(report_path, 'w') as f:
            json.dump(report, f, indent=2, default=str)
        print(f"    Relatório salvo: {report_path}")

        return report

print(f"{Fore.GREEN} Framework de qualidade definido!")

In [ ]:
# ============================================================
# ETAPA 5.2 — EXECUÇÃO DAS VERIFICAÇÕES DE QUALIDADE
# ============================================================

print(f"{Fore.YELLOW} Executando verificações de qualidade em todas as camadas...\n")

quality_reports = {}

# ─── Qualidade: Users Silver ───
dq_users = DataQualityCheck(df_users_silver, 'users', 'SILVER')
quality_reports['users'] = (
    dq_users
    .check_row_count(min_rows=5)
    .check_not_null(['user_id', 'name', 'email', 'username'])
    .check_unique(['user_id', 'email'])
    .check_regex('email', r'^[\w\.-]+@[\w\.-]+\.\w+$', 'formato email', severity='WARNING')
    .check_range('geo_lat', min_val=-90, max_val=90, severity='WARNING')
    .check_range('geo_lng', min_val=-180, max_val=180, severity='WARNING')
    .generate_report()
)

# ─── Qualidade: Posts Silver ───
dq_posts = DataQualityCheck(df_posts_silver, 'posts', 'SILVER')
quality_reports['posts'] = (
    dq_posts
    .check_row_count(min_rows=50)
    .check_not_null(['post_id', 'user_id', 'title'])
    .check_unique(['post_id'])
    .check_referential_integrity('user_id', df_users_silver, 'user_id')
    .check_range('title_word_count', min_val=1, max_val=50)
    .generate_report()
)

# ─── Qualidade: Gold User Profile ───
dq_gold = DataQualityCheck(df_gold_user_profile, 'user_profile', 'GOLD')
quality_reports['user_profile_gold'] = (
    dq_gold
    .check_not_null(['user_id', 'name', 'email', 'total_posts', 'total_tasks'])
    .check_unique(['user_id'])
    .check_range('completion_rate_pct', min_val=0, max_val=100)
    .check_range('total_posts', min_val=0)
    .generate_report()
)

In [ ]:
# ============================================================
# ETAPA 5.3 — SUMÁRIO GERAL DO PIPELINE
# ============================================================

print(f"\n{Fore.YELLOW}{'═'*60}")
print(f"{Fore.YELLOW}   SUMÁRIO FINAL DO PIPELINE ETL")
print(f"{Fore.YELLOW}{'═'*60}")

summary_data = []
for entity, report in quality_reports.items():
    score = report['score']
    icon = '🟢' if score >= 80 else ('🟡' if score >= 60 else '🔴')
    summary_data.append([
        entity.upper(),
        report['layer'],
        f"{icon} {score}%",
        report['summary']['passed'],
        report['summary']['failed'],
        report['summary']['warnings']
    ])

print(tabulate(
    summary_data,
    headers=['Entidade', 'Camada', 'Score', ' Pass', ' Fail', ' Warn'],
    tablefmt='rounded_outline'
))

print(f"\n{Fore.CYAN} Arquivos gerados no Data Lake:")
for dirpath, _, files in os.walk(BASE_PATH):
    layer = dirpath.split('/')[-1]
    for f in files:
        size = os.path.getsize(os.path.join(dirpath, f))
        print(f"   [{layer:15s}] {f} ({size:,} bytes)")

print(f"\n{Fore.GREEN} Pipeline ETL completo — Parte 1 (Pandas) finalizada!")
print(f"{Fore.GREEN}     Continue para a Parte 2: PySpark + Grandes Volumes")